In [1]:

import os
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# DAY 18: TEMPERATURE AND VELOCITY CORRELATION
# --------------------------------------------------

folders = glob.glob("combined_mantle_results*")

if not folders:
    raise FileNotFoundError(
        "Run the combined mantle convection model first."
    )

result_folder = max(folders, key=os.path.getmtime)
files = glob.glob(os.path.join(result_folder, "*.npz"))

if not files:
    raise FileNotFoundError("No .npz files found.")

output_folder = "day18_temperature_velocity"
os.makedirs(output_folder, exist_ok=True)

results = []
datasets = []

# --------------------------------------------------
# 1. LOAD AND ANALYSE EACH SIMULATION
# --------------------------------------------------

for filename in files:

    with np.load(filename) as data:

        required = ["ra", "x", "z", "T", "u", "w"]

        if not all(key in data for key in required):
            print("Skipping incomplete file:", filename)
            continue

        Ra = float(np.asarray(data["ra"]).item())

        x = data["x"].copy()
        z = data["z"].copy()
        T = data["T"].copy()
        u = data["u"].copy()
        w = data["w"].copy()

    # Check array dimensions
    if (
        T.ndim != 2
        or T.shape != u.shape
        or T.shape != w.shape
        or T.shape != (len(z), len(x))
    ):
        print("Skipping incompatible arrays:", filename)
        continue

    # Check finite values
    if not all(
        np.all(np.isfinite(a))
        for a in [x, z, T, u, w]
    ):
        print("Skipping non-finite data:", filename)
        continue

    if min(T.shape) < 3:
        print("Grid is too small:", filename)
        continue

    # ----------------------------------------------
    # TEMPERATURE ANOMALY
    # ----------------------------------------------

    # Horizontal mean at each height
    T_mean = np.mean(T, axis=1, keepdims=True)

    # Departure from the horizontal mean
    T_anomaly = T - T_mean

    # Velocity magnitude
    speed = np.sqrt(u**2 + w**2)

    # Ignore outer boundary cells for correlation
    Ta = T_anomaly[1:-1, 1:-1]
    wa = w[1:-1, 1:-1]

    # Correlation requires variation in both variables
    if np.std(Ta) > 1e-12 and np.std(wa) > 1e-12:
        correlation = np.corrcoef(
            Ta.ravel(), wa.ravel()
        )[0, 1]
    else:
        correlation = np.nan

    # Correlation between temperature anomaly and
    # vertical velocity: diagnostic, not proof of causation
    rms_anomaly = np.sqrt(np.mean(T_anomaly**2))
    max_speed = np.max(speed)

    results.append([
        Ra,
        correlation,
        rms_anomaly,
        max_speed
    ])

    datasets.append({
        "Ra": Ra,
        "x": x,
        "z": z,
        "T": T,
        "T_anomaly": T_anomaly,
        "u": u,
        "w": w,
        "speed": speed,
        "correlation": correlation
    })

    print("\nRayleigh number:", Ra)
    print("Temperature anomaly RMS:", rms_anomaly)
    print("Maximum speed:", max_speed)
    print("Correlation of T anomaly and vertical velocity:",
          correlation)

if not datasets:
    raise ValueError("No valid simulation datasets were found.")

datasets.sort(key=lambda item: item["Ra"])
results.sort(key=lambda row: row[0])

# --------------------------------------------------
# 2. PLOT TEMPERATURE ANOMALIES
# --------------------------------------------------

n = len(datasets)

fig, axes = plt.subplots(
    1, n,
    figsize=(5 * n, 4),
    squeeze=False
)

for i, item in enumerate(datasets):

    ax = axes[0, i]

    limit = np.max(np.abs(item["T_anomaly"]))
    limit = max(limit, 1e-12)

    contour = ax.contourf(
        item["x"],
        item["z"],
        item["T_anomaly"],
        levels=np.linspace(-limit, limit, 21),
        cmap="coolwarm",
        extend="both"
    )

    ax.set_title(f"Ra = {item['Ra']:g}")
    ax.set_xlabel("Horizontal position, x")
    ax.set_ylabel("Height, z")

fig.colorbar(
    contour,
    ax=axes.ravel().tolist(),
    label="Temperature anomaly",
    pad=0.03
)

fig.suptitle("Day 18: Temperature anomalies")

fig.subplots_adjust(
    top=0.82,
    bottom=0.15,
    left=0.08,
    right=0.90,
    wspace=0.35
)

fig.savefig(
    os.path.join(
        output_folder,
        "day18_temperature_anomalies.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close(fig)

# --------------------------------------------------
# 3. PLOT VERTICAL VELOCITY
# --------------------------------------------------

fig, axes = plt.subplots(
    1, n,
    figsize=(5 * n, 4),
    squeeze=False
)

for i, item in enumerate(datasets):

    ax = axes[0, i]

    limit = np.max(np.abs(item["w"]))
    limit = max(limit, 1e-12)

    contour = ax.contourf(
        item["x"],
        item["z"],
        item["w"],
        levels=np.linspace(-limit, limit, 21),
        cmap="coolwarm",
        extend="both"
    )

    ax.set_title(f"Ra = {item['Ra']:g}")
    ax.set_xlabel("Horizontal position, x")
    ax.set_ylabel("Height, z")

fig.colorbar(
    contour,
    ax=axes.ravel().tolist(),
    label="Vertical velocity, w",
    pad=0.03
)

fig.suptitle("Day 18: Vertical velocity fields")

fig.subplots_adjust(
    top=0.82,
    bottom=0.15,
    left=0.08,
    right=0.90,
    wspace=0.35
)

fig.savefig(
    os.path.join(
        output_folder,
        "day18_vertical_velocity.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close(fig)

# --------------------------------------------------
# 4. SCATTER PLOT
# --------------------------------------------------

fig, axes = plt.subplots(
    1, n,
    figsize=(5 * n, 4),
    squeeze=False
)

for i, item in enumerate(datasets):

    ax = axes[0, i]

    Ta = item["T_anomaly"][1:-1, 1:-1].ravel()
    wa = item["w"][1:-1, 1:-1].ravel()

    ax.scatter(
        Ta, wa,
        s=8,
        alpha=0.5
    )

    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)

    ax.set_xlabel("Temperature anomaly")
    ax.set_ylabel("Vertical velocity")

    ax.set_title(
        f"Ra = {item['Ra']:g}\n"
        f"Correlation = {item['correlation']:.3f}"
    )

fig.suptitle("Day 18: Temperature anomaly versus vertical flow")

fig.subplots_adjust(
    top=0.80,
    bottom=0.18,
    left=0.08,
    right=0.98,
    wspace=0.40
)

fig.savefig(
    os.path.join(
        output_folder,
        "day18_temperature_velocity_scatter.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close(fig)

# --------------------------------------------------
# 5. CORRELATION VERSUS RAYLEIGH NUMBER
# --------------------------------------------------

Ra_values = [row[0] for row in results]
correlations = [row[1] for row in results]

plt.figure(figsize=(7, 5))

plt.plot(
    Ra_values,
    correlations,
    "o-"
)

plt.axhline(0, color="black", linestyle="--")

plt.xlabel("Rayleigh number")
plt.ylabel("Correlation coefficient")
plt.title("Day 18: Temperature-flow correlation")
plt.ylim(-1.05, 1.05)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        output_folder,
        "day18_correlation_vs_Ra.png"
    ),
    dpi=200
)

plt.close()

# --------------------------------------------------
# 6. SAVE CSV SUMMARY
# --------------------------------------------------

csv_path = os.path.join(
    output_folder,
    "day18_correlation_summary.csv"
)

with open(csv_path, "w", newline="") as file:

    writer = csv.writer(file)

    writer.writerow([
        "Rayleigh_number",
        "Correlation_T_anomaly_vertical_velocity",
        "RMS_temperature_anomaly",
        "Maximum_speed"
    ])

    writer.writerows(results)

# --------------------------------------------------
# 7. FINISH
# --------------------------------------------------

print("\n" + "=" * 45)
print("DAY 18 ANALYSIS COMPLETED")
print("=" * 45)

print("Input folder:", result_folder)
print("Output folder:", output_folder)
print("Summary CSV:", csv_path)

print("\nGenerated plots:")
print("1. day18_temperature_anomalies.png")
print("2. day18_vertical_velocity.png")
print("3. day18_temperature_velocity_scatter.png")
print("4. day18_correlation_vs_Ra.png")
print("5. day18_correlation_summary.csv")



Rayleigh number: 1000.0
Temperature anomaly RMS: 0.11202641023312412
Maximum speed: 261.5621970066285
Correlation of T anomaly and vertical velocity: 0.9820315094148379

Rayleigh number: 2500.0
Temperature anomaly RMS: 0.10001147380793037
Maximum speed: 564.3629566969514
Correlation of T anomaly and vertical velocity: 0.9229786359791136

Rayleigh number: 500.0
Temperature anomaly RMS: 0.1137504364472756
Maximum speed: 133.09757056314587
Correlation of T anomaly and vertical velocity: 0.972090545507238

Rayleigh number: 5000.0
Temperature anomaly RMS: 0.09899473802759762
Maximum speed: 1064.8532313417584
Correlation of T anomaly and vertical velocity: 0.8480133662796124

DAY 18 ANALYSIS COMPLETED
Input folder: combined_mantle_results_2
Output folder: day18_temperature_velocity
Summary CSV: day18_temperature_velocity\day18_correlation_summary.csv

Generated plots:
1. day18_temperature_anomalies.png
2. day18_vertical_velocity.png
3. day18_temperature_velocity_scatter.png
4. day18_correla